# Chapter 0 · Foundations for AI: from tokens to reinforcement learning

Pages 0, 0A, 0B and 0C of the textbook, run in code: how a model reads text, writes it one token at a time, is built as a transformer, is trained, and is shaped by reinforcement learning. Open **one** notebook at the start and keep it: every section matches a page, in order, and each can be run on its own after the setup cell.

**No API key and nothing to download.** Everything runs on numpy, pandas and matplotlib, on the Champaign Capital Research figures used throughout the book. Where a real model would be needed, a small stand-in is used and labelled as one.

| Section | Page | What you run |
|---|---|---|
| 1 | 0.2 | Split a sentence into tokens |
| 2 | 0.3 to 0.6 | Predict the next word, change the temperature, fact-check the result |
| 3 | 0A | Attention, and a mixture-of-experts router |
| 4 | 0B | One training step, gradient descent, Adam |
| 5 | 0C | Reinforcement learning with two different scorers |

Run the first cell, then any section.

In [ ]:
import re, random
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)             # fixed seed: everyone gets the same numbers

# The firm's figures, the same as the textbook's tools.py.
FINANCIALS = {
    "DE":   {"name": "Deere",       "revenue": 13.8e9, "yoy": 0.064},
    "CAT":  {"name": "Caterpillar", "revenue": 16.9e9, "yoy": 0.031},
    "NVDA": {"name": "NVIDIA",      "revenue": 52.4e9, "yoy": 0.58},
    "AAPL": {"name": "Apple",       "revenue": 96.1e9, "yoy": 0.05},
    "MSFT": {"name": "Microsoft",   "revenue": 76.3e9, "yoy": 0.17},
}
print("ready")

## 1. Tokens: what the model reads (page 0.2)

A model does not read letters or words but **tokens**: pieces from a fixed list, learned from data. Common words are one piece; rarer names split into several. Cost, speed and the context window are all counted in tokens.

This tokenizer uses a hand-made list of 40 pieces. A real one has about 100,000. The rule is the real one: at each position take the longest piece that fits.

In [ ]:
VOCAB = [" revenue", " growth", " grew", " faster", " than", " the", " last", " quarter", " year", " to", " of",
         " is", " and", " a", " from", " base", " smaller", " larger", " client", " memo", " prompt",
         "Deere", " Deere", "Apple", " Apple", "NVIDIA", " NVIDIA", "Micro", " Micro", "soft",
         "Cater", " Cater", "pillar", "'s", "%", "$", "B", ".", ",", "?"]
_VOCAB = sorted(set(VOCAB), key=len, reverse=True)

def tokenize(text):
    pieces, i = [], 0
    while i < len(text):
        piece = next((v for v in _VOCAB if text.startswith(v, i)), text[i])
        pieces.append(piece); i += len(piece)
    return pieces

def show_tokens(text):
    pieces = tokenize(text)
    print(f"{len(text.split())} words -> {len(pieces)} tokens")
    print(" ".join(f"[{p.replace(' ', '·')}]" for p in pieces))

show_tokens("Deere grew revenue faster than Caterpillar last quarter.")
show_tokens("Microsoft's revenue grew 17.0% to $76.3B.")

## 2. Writing one token at a time (pages 0.3 to 0.6)

A language model predicts the next token, appends it, and predicts again. This toy version predicts the next **word** from counts over thirteen firm sentences. A real model uses a neural network trained on trillions of tokens, but the loop is the same: look at what came before, get odds for what comes next, pick one.

**Temperature** reshapes the odds. Low: the favourite gets even more likely. High: the long shots get a real chance.

In [ ]:
CORPUS = [
    "Deere grew revenue faster than Caterpillar .", "Deere grew revenue 6.4% to $13.8B .",
    "Caterpillar grew revenue 3.1% to $16.9B .", "NVIDIA grew revenue faster than Apple .",
    "NVIDIA grew revenue 58.0% to $52.4B .", "Apple grew revenue 5.0% to $96.1B .",
    "Microsoft grew revenue 17.0% to $76.3B .", "Microsoft grew revenue faster than Apple .",
    "Deere grew revenue faster than Caterpillar from a smaller base .", "NVIDIA is growing faster than Microsoft .",
    "Deere is growing faster than Caterpillar .", "Caterpillar is larger than Deere by revenue .",
    "Apple is larger than NVIDIA .",
]
FOLLOWS = defaultdict(Counter)
for sentence in CORPUS:
    words = sentence.split()
    for a, b in zip(words, words[1:]):
        FOLLOWS[a][b] += 1

def next_word_probs(word, temperature=1.0):
    counts = FOLLOWS.get(word)
    if not counts: return []
    if temperature <= 0:
        best = max(counts.values()); tops = [w for w, c in counts.items() if c == best]
        return [(w, 1 / len(tops) if w in tops else 0.0) for w, _ in counts.most_common()]
    weights = {w: c ** (1 / temperature) for w, c in counts.items()}
    total = sum(weights.values())
    return sorted(((w, x / total) for w, x in weights.items()), key=lambda wp: -wp[1])

for t in (0.3, 1.0, 2.0):
    print(f"temperature {t}:", [(w, round(p, 2)) for w, p in next_word_probs("revenue", t)])

In [ ]:
def generate(start, temperature=0.0, seed=0, max_words=12):
    r, words = random.Random(seed), [start]
    while words[-1] != "." and len(words) < max_words:
        probs = next_word_probs(words[-1], temperature)
        if not probs: break
        words.append(probs[0][0] if temperature <= 0 else
                     r.choices([w for w, _ in probs], weights=[p for _, p in probs])[0])
    return " ".join(words)

def fact_check(sentence):
    row = next((v for v in FINANCIALS.values() if v["name"] == sentence.split()[0]), None)
    growth = re.search(r"(\d+\.\d)%", sentence); revenue = re.search(r"\$(\d+\.\d)B", sentence)
    if not row or not (growth or revenue): return "no figures to check"
    wrong = []
    if growth and abs(float(growth.group(1)) - row["yoy"] * 100) > 0.05:
        wrong.append(f"growth was {row['yoy']*100:.1f}%, not {growth.group(1)}%")
    if revenue and abs(float(revenue.group(1)) - row["revenue"] / 1e9) > 0.05:
        wrong.append(f"revenue was ${row['revenue']/1e9:.1f}B, not ${revenue.group(1)}B")
    return "matches the records" if not wrong else "WRONG: " + " and ".join(wrong)

print("temperature 0:", generate("Deere"))
print()
for seed in range(6):
    s = generate("Deere", temperature=1.5, seed=seed)
    print(f"{s:60} -> {fact_check(s)}")

Nothing in the loop checks whether a sentence is true. At a high temperature the toy model sometimes writes a fluent sentence with the wrong figure. That is **hallucination** in miniature (page 0.6), and it is why later chapters give the model tools to look figures up.

## 3. Transformers and mixture of experts (page 0A)

Each token becomes a list of numbers (an **embedding**). Attention then lets every token look at the tokens before it. Each token makes a **query** (what am I looking for?), a **key** (what do I offer?) and a **value** (what do I pass on?), and takes in more of the values whose keys match its query.

The weights below are random, so the patterns mean nothing. The arithmetic is the real one.

In [ ]:
tokens = ["Deere", "grew", "revenue", "faster", "than", "Caterpillar"]
n, d = len(tokens), 8
X = np.stack([rng.normal(size=d) for _ in tokens])          # one embedding per token
Wq, Wk, Wv = (rng.normal(size=(d, d)) for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv

scores = Q @ K.T / np.sqrt(d)
scores[np.triu(np.ones((n, n), dtype=bool), k=1)] = -np.inf  # a token cannot see later tokens
weights = np.exp(scores - scores.max(axis=1, keepdims=True))
weights /= weights.sum(axis=1, keepdims=True)                # softmax: each row adds up to 1
output = weights @ V

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(weights, cmap="Blues")
ax.set_xticks(range(n)); ax.set_xticklabels(tokens, rotation=45, ha="right")
ax.set_yticks(range(n)); ax.set_yticklabels(tokens)
ax.set_xlabel("looked at"); ax.set_ylabel("reading"); ax.set_title("Attention weights (random model)")
plt.tight_layout(); plt.show()

In [ ]:
# Every token looks at every earlier token, so the table grows with the square of the length.
lengths = (100, 200, 400, 800)
cells_ = [L * (L + 1) // 2 for L in lengths]
pd.DataFrame({"tokens": lengths, "cells in the attention table": cells_,
              "compared with 100 tokens": [round(c / cells_[0], 1) for c in cells_]})

In [ ]:
# Mixture of experts: a router picks the top 2 of 8 experts for each token.
n_experts, top_k = 8, 2
router = rng.normal(size=(d, n_experts))
chosen = np.argsort(-(output @ router), axis=1)[:, :top_k]
print(pd.DataFrame({"token": tokens, "experts used": [sorted(int(e) for e in row) for row in chosen]}))
print(f"\nEach token uses {top_k} of {n_experts} experts: {top_k / n_experts:.0%} of the layer works per token.")

## 4. How a model is trained (page 0B)

Every stage of training repeats one loop: **predict** the next token, **compare** with what really came (the error, or loss), work out how much each weight contributed (**backpropagation**), and let the **optimizer** nudge every weight to shrink the error.

**One training step.** A tiny model has one number per candidate next word. We train it to make "grew" likely after "Deere's revenue":

In [ ]:
words = ["grew", "fell", "is", "."]
logits = np.zeros(4)                        # the model's four weights, all equal at the start
target = 0                                  # the word that really came next: "grew"
lr = 0.5
history = []
for step in range(12):
    p = np.exp(logits) / np.exp(logits).sum()          # predict
    loss = -np.log(p[target])                          # compare: how wrong was it?
    grad = p.copy(); grad[target] -= 1                 # go back: the slope for each weight
    logits -= lr * grad                                # nudge every weight downhill
    history.append(loss)
print("probability of 'grew' after 12 steps:", round(float(p[target]), 2))
plt.plot(history); plt.xlabel("training step"); plt.ylabel("error"); plt.title("The error falls"); plt.show()

**Gradient descent** takes a small step downhill: new weight = old weight − learning rate × gradient. On a long narrow valley (steep in one direction, gentle in the other) plain steps bounce from wall to wall. **Momentum** keeps part of the last step so the bounces cancel. **Adam** gives every weight its own step size, using running averages of the slope and its size.

Race them on the same valley, error = 0.5·(x² + 25·y²):

In [ ]:
def grad_f(w): return np.array([w[0], 25 * w[1]])
def loss_f(w): return 0.5 * (w[0] ** 2 + 25 * w[1] ** 2)

def run(method, steps=60, lr=0.05):
    w = np.array([4.0, 1.0]); v = np.zeros(2); m = np.zeros(2); s = np.zeros(2); out = [loss_f(w)]
    for t in range(1, steps + 1):
        g = grad_f(w)
        if method == "gradient descent":
            w = w - lr * g
        elif method == "momentum":
            v = 0.8 * v + g; w = w - 0.03 * v
        else:                                              # Adam
            m = 0.9 * m + 0.1 * g; s = 0.999 * s + 0.001 * g ** 2
            w = w - 0.3 * (m / (1 - 0.9 ** t)) / (np.sqrt(s / (1 - 0.999 ** t)) + 1e-8)
        out.append(loss_f(w))
    return out

for name in ("gradient descent", "momentum", "Adam"):
    curve = run(name); plt.plot(curve, label=name)
    reached = next((i for i, e in enumerate(curve) if e < 0.05), None)
    print(f"{name:17} error after 60 steps: {curve[-1]:.5f}   steps to get below 0.05: {reached}")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("error (log scale)"); plt.legend(); plt.show()

Momentum and Adam both beat plain descent here. Adam's edge is that one setting copes when weights have very different slopes, so it needs less tuning. Try `lr=0.07` for plain gradient descent (the steep direction bounces hard), then `lr=0.09` (it flies off). Adam's cost: two extra numbers per weight for the whole of training, which is why training needs far more memory than answering (page 0E).

## 5. Pre-, mid- and post-training, and reinforcement learning (page 0C)

Pre-training reads the web; mid-training reads more specialised text; post-training teaches the model to answer as people want, ending with **reinforcement learning**: the model writes several answers, each is **scored**, and answers that beat the average become more likely.

The score comes from somewhere. Below, a model chooses between four answer *styles* for "What was Deere's revenue growth?". Train it with two different scorers:

In [ ]:
styles = ["confident, wrong figure", "hedged, right figure", "confident, right figure", "refuses to answer"]
# Scorer 1: reviewers' preferences. They cannot check the figure, so both confident answers score alike.
people = np.array([0.95, 0.4, 0.95, 0.1])
# Scorer 2: a check that can be run against the firm's data. Only the right figure passes.
checker = np.array([0.0, 1.0, 1.0, 0.0])

def train(reward, rounds=200, lr=0.5, seed=1):
    r = np.random.default_rng(seed); prefs = np.zeros(4)
    for _ in range(rounds):
        p = np.exp(prefs) / np.exp(prefs).sum()
        picks = r.choice(4, size=4, p=p)                 # four answers per round
        scores = reward[picks]
        advantage = scores - scores.mean()               # better than this round's average?
        for k, a in zip(picks, advantage):
            prefs[k] += lr * a * (1 - p[k])              # make it more likely, if it beat the average
    return np.exp(prefs) / np.exp(prefs).sum()

for name, reward in (("people's preferences", people), ("a fact check", checker)):
    p = train(reward)
    print(f"Scored by {name}:")
    for s, x in zip(styles, p): print(f"   {x:5.0%}  {s}")

Reinforcement learning teaches a model **what gets rewarded, not what is true**. With a fact-checking scorer the wrong-figure style dies out. With reviewers' preferences alone it survives, because a wrong answer that sounds sure of itself scores as well as a right one.

## Try it

1. Section 1: tokenise a sentence from your own work. Which words split into several tokens?
2. Section 2: run 20 seeds at temperature 1.5, then at 0.5. How many false sentences did each produce?
3. Section 3: change `top_k` to 1, then 4. How much of the layer works per token?
4. Section 4: change the learning rate in the gradient-descent race. When does it stop converging?
5. Section 5: change `people` so reviewers like hedged answers. What does the model learn?